In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# --- Proje Sabitleri ---
DATA_DIR = "../../data/prepared-data" 
TEST_DIR = os.path.join(DATA_DIR, "test")

# Eğitilmiş TensorFlow (.h5) modellerinin klasörü
MODELS_DIR = "../../models/" 

NUM_CLASSES = 8
IMG_SIZE = (224, 224)
BATCH_SIZE = 64 

# GPU kontrolü
print("Kullanılabilir GPU Sayısı: ", len(tf.config.experimental.list_physical_devices('GPU')))

Kullanılabilir GPU Sayısı:  1


In [ ]:
# Test veri setini yüklüyoruz.
# *** shuffle=False OLMASI ZORUNLUDUR ***
# Bu, etiket sırasının tahmin sırasıyla eşleşmesini garantiler.
test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical', 
    shuffle=False 
)

class_names = test_dataset.class_names
print(f"Sınıflar: {class_names}")
print(f"Test verisi: {len(test_dataset.file_paths)} görüntü")

# --- Gerçek Etiketleri (y_true) Al ---
# dataset.labels parametresi 'categorical' modda çalışmayabilir,
# en garantili yol, dataseti iterate etmektir.
y_true = np.concatenate([np.argmax(y, axis=-1) for x, y in test_dataset], axis=0)
print(f"Gerçek etiketler (y_true) başarıyla alındı. Toplam: {len(y_true)}")

Found 808 files belonging to 8 classes.
Sınıflar: ['dyed-lifted-polyps', 'dyed-resection-margins', 'esophagitis', 'normal-cecum', 'normal-pylorus', 'normal-z-line', 'polyps', 'ulcerative-colitis']
Test verisi: 808 görüntü
Gerçek etiketler (y_true) başarıyla alındı. Toplam: 808


: 

In [ ]:
# Tüm sonuçları saklamak için bir liste
evaluation_results = []

# Model klasöründeki tüm .h5 dosyalarını bul
model_files = [f for f in os.listdir(MODELS_DIR) if f.endswith('.h5')]

print(f"Toplam {len(model_files)} adet eğitilmiş TensorFlow modeli bulundu.")

for model_file in model_files:
    print(f"\n{'='*20}")
    print(f"DEĞERLENDİRİLİYOR: {model_file}")
    print(f"{'='*20}")
    
    model_path = os.path.join(MODELS_DIR, model_file)
    
    try:
        # --- 1. Modeli Yükle ---
        # Keras'ın güzelliği: model.save() ile kaydettiğimiz .h5 dosyası,
        # mimariyi, ağırlıkları ve bizim eklediğimiz ön işleme pipeline'ını
        # (boş pipeline veya ResNet preprocess_input) birlikte içerir.
        model = tf.keras.models.load_model(model_path)
        print(f"Model başarıyla yüklendi: {model_file}")

        # --- 2. Tahminleri Topla ---
        # model.predict(), veri seti üzerinde döngü kurar ve tahminleri döner
        print("Test seti üzerinde tahminler yapılıyor...")
        y_pred_probs = model.predict(test_dataset)
        
        # Olasılıklardan (probs) en yüksek skorlu sınıfın indeksini al (0, 1, 2...)
        y_pred_classes = np.argmax(y_pred_probs, axis=1)

        # --- 3. Metrikleri Hesapla ---
        accuracy = accuracy_score(y_true, y_pred_classes)
        report_dict = classification_report(y_true, y_pred_classes, target_names=class_names, output_dict=True)
        
        print(f"\n--- {model_file} Sonuçları ---")
        print(f"Genel Doğruluk (Accuracy): {accuracy * 100:.2f}%")
        print("\nSınıflandırma Raporu (Classification Report):")
        # Raporu konsola güzel bir formatta yazdır
        print(classification_report(y_true, y_pred_classes, target_names=class_names))
        
        # Sonuçları ana tablo için kaydet
        evaluation_results.append({
            "Model": model_file.replace('.h5', ''),
            "Accuracy": accuracy,
            "F1-Score (Weighted)": report_dict['weighted avg']['f1-score'],
            "Precision (Weighted)": report_dict['weighted avg']['precision'],
            "Recall (Weighted)": report_dict['weighted avg']['recall']
        })

        # --- 4. Karmaşıklık Matrisini (Confusion Matrix) Çizdir ---
        cm = confusion_matrix(y_true, y_pred_classes)
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=class_names, yticklabels=class_names)
        plt.title(f'Karmaşıklık Matrisi - {model_file}')
        plt.xlabel('Tahmin Edilen (Predicted)')
        plt.ylabel('Gerçek (True)')
        plt.show()

    except Exception as e:
        print(f"!!! MODEL {model_file} YÜKLENİRKEN/DEĞERLENDİRİLİRKEN HATA OLUŞTU: {e}")


print("\n\n--- DEĞERLENDİRME TAMAMLANDI ---")

Toplam 12 adet eğitilmiş TensorFlow modeli bulundu.

DEĞERLENDİRİLİYOR: efficientnetv2b0_augmented_FIXED.h5
Model başarıyla yüklendi: efficientnetv2b0_augmented_FIXED.h5
Test seti üzerinde tahminler yapılıyor...


In [ ]:
# Sonuçları Pandas DataFrame'e çevir
results_df = pd.DataFrame(evaluation_results)
results_df = results_df.set_index("Model") # Model ismini index yap

# F1-Skoruna göre sırala
results_df = results_df.sort_values(by="F1-Score (Weighted)", ascending=False)

print("--- NİHAİ MODEL KARŞILAŞTIRMA TABLOSU (Test Seti Sonuçları) ---")
print(results_df.to_markdown(floatfmt=".4f"))